# Classical models: Pokémon and attack identities

Only **logistic regression and random forest**. Four initial comparisons, then a bounded three-candidate LR scaling/regularization follow-up after the first identity encoding overfit. Five identical grouped development folds; previous holdout and duplicates excluded. Default: review saved results.


In [ ]:
from pathlib import Path
import json, csv, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pokemon_challenge").is_dir())
sys.path.insert(0, str(ROOT / "notebooks/leonard"))
RUN_NEW_EXPERIMENT = False
RUN_FOLDER = ROOT / "outputs/leonard/20261102_124639_898979_classical_identity_followup"
if RUN_NEW_EXPERIMENT:
    import run_classical_identity_experiment as initial
    import run_identity_scaling_followup as followup
    followup.INITIAL = initial.run()
    RUN_FOLDER = followup.run_followup()
print("Artifacts:", RUN_FOLDER)


## Features and encoding

The 289 numeric inputs retain the earlier summaries and boost threat. New inputs describe species membership/last HP, move frequency, recent moves, recency, attack types and seven categorical identities. Team membership is unordered. Vocabularies are learned on training folds only.

Initial scaling standardized every nonconstant input, which hurt LR with roughly 4,250 features. The follow-up scales baseline numeric inputs only, keeping rare identity indicators/rates on their original scales; C=.01/.1/1 are compared. This follow-up is adaptive and its CV gain is exploratory.


In [ ]:
with (RUN_FOLDER / "cv_results.csv").open() as stream:
    rows = list(csv.DictReader(stream))
for row in rows:
    print(f"{row['candidate']:38s} {100*float(row['cv_accuracy']):.3f}%")


## Selected model and limitations

**LR, C=.01, baseline-only scaling: 67.259% CV**. Numeric LR: 66.533%; numeric RF: 66.546%. Prior boosting reference: 67.234%, only 0.025 pp lower, so the two are practically tied. This does not demonstrate a higher Kaggle score.

Final LR is refitted on all 10,000 labels and generates 5,000 predictions. No holdout re-evaluation, boosting or ensemble. Twelve tests and independent checks pass, including serialized-model reproduction on raw test observations. Last-seen roster estimates and repeated development selection remain limitations.


In [ ]:
report = json.loads((RUN_FOLDER / "run_summary.json").read_text())
validation = json.loads((RUN_FOLDER / "validation_report.json").read_text())
print(json.dumps(report, indent=2))
print(json.dumps(validation, indent=2))
